# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -q mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")
print(f"Published: {metadata.datePublished}")
print(f"Identifier: {metadata.identifier}")
print(f"License: {metadata.license}")
print(f"Citation: {getattr(metadata, 'citeAs', 'N/A')}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List all available record sets and their fields using their @id

record_sets = list(dataset.record_sets)  # List of mlcroissant.RecordSet
print(f"Dataset contains {len(record_sets)} record sets.")

for record_set in record_sets:
    print(f"\nRecord Set: {record_set.name}")
    print(f"  @id: {record_set.id}")
    # List Fields
    print("  Fields:")
    for field in record_set.fields:
        print(f"    - Name: {field.name}, @id: {field.id}, DataType: {field.data_type}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# For demonstration, select the main record set, usually the one containing protein data.
# Review names and ids from previous cell!

main_record_set_id = None
for rs in record_sets:
    # Try to find the most likely main record set by field names
    field_names = [f.name.lower() for f in rs.fields]
    # Heuristic: seek a record set with fields like 'accession', 'description', etc.
    main_fields = {'accession', 'description', 'coverage', 'peptide', 'sequence', 'abundance'}
    if any(any(x in name for x in main_fields) for name in field_names):
        main_record_set_id = rs.id
        break

assert main_record_set_id is not None, "No main record set found. Please check the dataset."

print(f"Selected main record set @id: {main_record_set_id}")

# Optionally load all record sets
record_set_ids = [rs.id for rs in record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)

# Show available columns in main record set
main_df = dataframes[main_record_set_id]
print(f"Columns in main record set ({main_record_set_id}):\n{main_df.columns.tolist()}")
main_df.head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# Choose a numeric field for demonstration (e.g. coverage percentage, peptide_count, abundance, molecular_weight)
# We'll auto-detect a numeric column

numeric_field = None
for col in main_df.columns:
    # Use a simple heuristic
    if (('coverage' in col.lower() or 'abundance' in col.lower() or 'peptide' in col.lower() or 'mw' in col.lower() or 'weight' in col.lower() or 'count' in col.lower()) \
        and pd.api.types.is_numeric_dtype(main_df[col])):
        numeric_field = col
        break
# Fallback: first numeric column
if numeric_field is None:
    for col in main_df.columns:
        if pd.api.types.is_numeric_dtype(main_df[col]):
            numeric_field = col
            break

if numeric_field is None:
    raise ValueError("No numeric column found for EDA.")

print(f"Selected numeric field: '{numeric_field}'")

# Filtering
threshold = main_df[numeric_field].mean()
filtered_df = main_df[main_df[numeric_field] > threshold].copy()
print(f"Filtered records with '{numeric_field}' > mean ({threshold:.2f}): {len(filtered_df)} rows")
display(filtered_df.head())

# Normalizing the numeric field
filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"Normalized '{numeric_field}' for filtered records:")
display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Group by a likely applicable field (heuristic)
group_field = None
for col in main_df.columns:
    if any(x in col.lower() for x in ['category', 'condition', 'group', 'type']) and main_df[col].nunique() > 1:
        group_field = col
        break
if group_field is not None:
    print(f"Grouping by field: '{group_field}'")
    grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
    display(grouped_df.head())
else:
    print("No appropriate group field found for grouping.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the numeric field
plt.figure(figsize=(8,5))
sns.histplot(main_df[numeric_field].dropna(), kde=True, bins=30)
plt.title(f"Distribution of '{numeric_field}'")
plt.xlabel(numeric_field)
plt.ylabel('Count')
plt.show()

# Scatter plot for two biggest numeric fields, if possible
other_numeric = [col for col in main_df.columns if pd.api.types.is_numeric_dtype(main_df[col]) and col != numeric_field]
if other_numeric:
    x_field = numeric_field
    y_field = other_numeric[0]
    plt.figure(figsize=(7,5))
    sns.scatterplot(x=main_df[x_field], y=main_df[y_field])
    plt.xlabel(x_field)
    plt.ylabel(y_field)
    plt.title(f"Scatter plot: {x_field} vs. {y_field}")
    plt.show()
else:
    print("No other numeric field found for scatter plot.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- We've loaded the *Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells* dataset using the Croissant schema via `mlcroissant`.
- We reviewed metadata and automatically explored available record sets and fields using their `@id` values for reproducibility.
- Data was loaded into DataFrames, and basic exploratory analysis conducted on a key numeric field (e.g., protein abundance, coverage, or similar).
- Visualizations illustrate distribution and quantitative structure of the protein data, aiding further biological or computational analyses.
- For more advanced EDA, refer to specific field descriptions using the Croissant metadata in this notebook.